In [ ]:
import warnings
warnings.simplefilter(action='ignore')
import pycisTopic
pycisTopic.__version__
import os

projDir = '/home/spalo/lu2025-12-50/spalo/scenicplus_storage/'
outDir = projDir + 'outs/merged/'

In [ ]:
import mudata
scplus_mdata = mudata.read(outDir + 'scplusmdata.h5mu')

In [ ]:
scplus_mdata.uns.keys()

In [ ]:
from scenicplus.RSS import (regulon_specificity_scores, plot_rss)

In [ ]:
rss = regulon_specificity_scores(
    scplus_mudata = scplus_mdata,
    variable = "scRNA_counts:scenic_ident",
    modalities = ["direct_gene_based_AUC", "direct_region_based_AUC"]
)

In [ ]:
rss

In [ ]:
rss.to_csv(outDir + 'rss_gene_region.csv')

In [ ]:
plot_rss(
    data_matrix = rss,
    top_n = 4,
    num_columns = 4
)

In [ ]:
print(rss.loc['BMHSC'].sort_values(ascending = False)[0:10].index)
print(rss.loc['FLHSC'].sort_values(ascending = False)[0:10].index)

In [ ]:
from scenicplus.scenicplus_class import mudata_to_scenicplus
#make copies of the scplus object since the log2FC in the nx table function permanently alters the counts matrices 
scplus_obj1 = mudata_to_scenicplus(mdata = scplus_mdata.copy(), path_to_cistarget_h5 = outDir + "ctx_results.hdf5",
    path_to_dem_h5 = outDir + "dem_results.hdf5")
scplus_obj2 = mudata_to_scenicplus(mdata = scplus_mdata.copy(), path_to_cistarget_h5 = outDir + "ctx_results.hdf5",
    path_to_dem_h5 = outDir + "dem_results.hdf5")

In [ ]:
scplus_obj1.uns.keys()

In [ ]:
scplus_mdata.uns.keys()

In [ ]:
scplus_obj1.uns['direct_e_regulon_metadata'] = scplus_mdata.uns['direct_e_regulon_metadata'].copy()
scplus_obj2.uns['direct_e_regulon_metadata'] = scplus_mdata.uns['direct_e_regulon_metadata'].copy()

In [ ]:
import scenicplus

In [ ]:
from scenicplus.networks import create_nx_tables, create_nx_graph, plot_networkx, export_to_cytoscape

In [ ]:
scplus_obj1.uns['direct_e_regulon_metadata']

In [ ]:
direct_md = scplus_obj1.uns['direct_e_regulon_metadata']

In [ ]:
#filter for +/+ interactions
mask = direct_md['eRegulon_name'].str.contains('_\\+\\/\\+')
direct_md = direct_md[mask]
direct_md

In [ ]:
#subset for only signatures that are included in the filtered eregulon metadata
import pandas
df1 = rss[set(direct_md['Gene_signature_name']) & set(rss.columns)]
df2 = rss[set(direct_md['Region_signature_name']) & set(rss.columns)]
rss_act = pandas.concat([df1, df2], axis = 1)
rss_act

In [ ]:
BMHSC = rss_act.loc['BMHSC'].sort_values(ascending = False)[0:10]
print(BMHSC)
BMHSC = [str(col)[:col.rindex('_')] for col in BMHSC.index]
BMHSC

In [ ]:
FLHSC = rss_act.loc['FLHSC'].sort_values(ascending = False)[0:10]
print(FLHSC)
FLHSC = [str(col)[:col.rindex('_')] for col in FLHSC.index]
FLHSC

In [ ]:
bmhsc_filt = direct_md[direct_md['eRegulon_name'].isin(BMHSC)]
flhsc_filt = direct_md[direct_md['eRegulon_name'].isin(FLHSC)]
flhsc_filt

In [ ]:
scplus_obj1.uns['e_regulon_metadata_bmhsc'] = bmhsc_filt
scplus_obj2.uns['e_regulon_metadata_flhsc'] = flhsc_filt
scplus_obj2.uns['e_regulon_metadata_flhsc']

In [ ]:
from pycisTopic.diff_features import find_highly_variable_features
hvr_bm = find_highly_variable_features(scplus_obj1.to_df('ACC').loc[list(set(scplus_obj1.uns['e_regulon_metadata_bmhsc']['Region']))], n_top_features=800, plot = True)
hvg_bm = find_highly_variable_features(scplus_obj1.to_df('EXP')[list(set(scplus_obj1.uns['e_regulon_metadata_bmhsc']['Gene']))].T, n_top_features=300, plot = True)
hvr_fl = find_highly_variable_features(scplus_obj2.to_df('ACC').loc[list(set(scplus_obj2.uns['e_regulon_metadata_flhsc']['Region']))], n_top_features=800, plot = True)
hvg_fl = find_highly_variable_features(scplus_obj2.to_df('EXP')[list(set(scplus_obj2.uns['e_regulon_metadata_flhsc']['Gene']))].T, n_top_features=300, plot = True)

In [ ]:
nxtab_bmhsc = scenicplus.networks.create_nx_tables(
    scplus_obj = scplus_obj1,
    eRegulon_metadata_key ='e_regulon_metadata_bmhsc',
    subset_regions = hvr_bm,
    subset_genes = hvg_bm,
    add_differential_gene_expression = True,
    add_differential_region_accessibility = True,
    differential_variable = ['scenic_ident']
)
nxtab_bmhsc['Node']['Gene'].to_csv(outDir + 'BMHSC_nw_gene_tab.csv')
nxtab_bmhsc['Node']['Region'].to_csv(outDir + 'BMHSC_nw_region_tab.csv')

In [ ]:
scplus_obj1.X_EXP

In [ ]:
scplus_obj2.X_EXP

In [ ]:
hvg_bm 

In [ ]:
nxtab_flhsc = scenicplus.networks.create_nx_tables(
    scplus_obj = scplus_obj2,
    eRegulon_metadata_key ='e_regulon_metadata_flhsc',
    subset_regions = hvr_fl,
    subset_genes = hvg_fl,
    add_differential_gene_expression = True,
    add_differential_region_accessibility = True,
    differential_variable = ['scenic_ident']
)
nxtab_flhsc['Node']['Gene'].to_csv(outDir + 'FLHSC_nw_gene_tab.csv')
nxtab_flhsc['Node']['Region'].to_csv(outDir + 'FLHSC_nw_region_tab.csv')

In [ ]:
category_color_bmhsc = {'HOXB2': '#3C734A', 'KLF5': '#ab7560', 'ZNF385D': '#e2b3f5', 'ZBTB20' : '#c7c7c7', 'TBL1XR1': '#fcd63e', 
                        'HLF': '#fcab40', 'GATA3' : '#874E80', 'NFIA': '#AB3737', 'MDX1' : '#2333ffd6', 'FOS': '#ffb9cd'}

category_color_flhsc = {'HOXB8': '#b2c771', 'HOXB7': '#68b594', 'HLF': '#fcab40', 'HMGA2' : '#824A3C', 'MTF2': '#7e66e5de', 
                       'CHURC1': '#79CCD1', 'GATA3' : '#874E80', 'HOXB2' : '#3C734A'}

In [ ]:
nxtab_bmhsc['Node']['Region'].iloc[:, 2:10]

In [ ]:
gmin = nxtab_bmhsc['Node']['Gene'].iloc[:, 2:10].stack().mean() - nxtab_bmhsc['Node']['Gene'].iloc[:, 2:10].stack().std()
gmax = nxtab_bmhsc['Node']['Gene'].iloc[:, 2:10].stack().mean() + nxtab_bmhsc['Node']['Gene'].iloc[:, 2:10].stack().std()
rmin = nxtab_bmhsc['Node']['Region'].iloc[:, 2:10].stack().mean() - nxtab_bmhsc['Node']['Region'].iloc[:, 2:10].stack().std()
rmax = nxtab_bmhsc['Node']['Region'].iloc[:, 2:10].stack().mean() + nxtab_bmhsc['Node']['Region'].iloc[:, 2:10].stack().std()
rmax

In [ ]:
G, pos, edge_tables, node_tables = create_nx_graph(nxtab_bmhsc,
                   use_edge_tables = ['TF2R','R2G'],
                   color_edge_by = {'TF2R': {'variable' : 'TF', 'category_color' : category_color_bmhsc},
                                    'R2G': {'variable' : 'rho_R2G', 'continuous_color' : 'viridis', 'v_min': -1, 'v_max': 1}},
                   #transparency_edge_by =  {'R2G': {'variable' : 'importance_R2G', 'min_alpha': 0.1, 'v_min': 0}},
                   width_edge_by = {'R2G': {'variable' : 'importance_R2G', 'max_size' :  1.5, 'min_size' : 1}},
                   color_node_by = {'TF': {'variable': 'TF', 'category_color' : category_color_bmhsc},
                                    'Gene': {'variable': 'scenic_ident_Log2FC_BMHSC', 'continuous_color' : 'bwr', 'v_min': gmin, 'v_max': gmax},
                                    'Region': {'variable': 'scenic_ident_Log2FC_BMHSC', 'continuous_color' : 'viridis', 'v_min': rmin, 'v_max': rmax}},
                   #transparency_node_by =  {'Region': {'variable' : 'scenic_ident_Log2FC_BMHSC', 'min_alpha': 0.1},
                  #                  'Gene': {'variable' : 'scenic_ident_Log2FC_BMHSC', 'min_alpha': 0.1}},
                   size_node_by = {'TF': {'variable': 'fixed_size', 'fixed_size': 20},
                                    'Gene': {'variable': 'fixed_size', 'fixed_size': 10},
                                    'Region': {'variable': 'fixed_size', 'fixed_size': 8}},
                   shape_node_by = {'TF': {'variable': 'fixed_shape', 'fixed_shape': 'ellipse'},
                                    'Gene': {'variable': 'fixed_shape', 'fixed_shape': 'ellipse'},
                                    'Region': {'variable': 'fixed_shape', 'fixed_shape': 'diamond'}},
                   label_size_by = {'TF': {'variable': 'fixed_label_size', 'fixed_label_size': 9.0},
                                    'Gene': {'variable': 'fixed_label_size', 'fixed_label_size': 5.0},
                                    'Region': {'variable': 'fixed_label_size', 'fixed_label_size': 1.0}},
                   layout='kamada_kawai_layout',
                   scale_position_by=250)

scenicplus.networks.export_to_cytoscape(G, pos, out_file = outDir + 'cs_bm_hsc.cyjs')

In [ ]:
scenicplus.networks.plot_networkx(G, pos)

In [ ]:
# quick dirty way to make color scale bars since these seem to be omitted from the network plot wrapper function
import matplotlib.pyplot as plt
print(nxtab_bmhsc['Node']['Gene'].iloc[:, 3])
gvals = nxtab_bmhsc['Node']['Gene'].iloc[:, 3]
data = (gvals, gvals)

cmap = plt.cm.get_cmap("bwr")
plt.imshow(data, cmap, vmin = gmin, vmax = gmax)
plt.colorbar()
plt.savefig(outDir + "bm_hsc_genecolors.pdf", format = "pdf")  

In [ ]:
print(nxtab_bmhsc['Node']['Region'].iloc[:, 3])
gvals = nxtab_bmhsc['Node']['Region'].iloc[:, 3]
data = (gvals, gvals)

cmap = plt.cm.get_cmap("viridis")
plt.imshow(data, cmap, vmin = rmin, vmax = rmax)
plt.colorbar()
plt.savefig(outDir + "bm_hsc_regioncolors.pdf", format = "pdf")  

In [ ]:
gmin = nxtab_flhsc['Node']['Gene'].iloc[:, 2:10].stack().mean() - nxtab_flhsc['Node']['Gene'].iloc[:, 2:10].stack().std()
gmax = nxtab_flhsc['Node']['Gene'].iloc[:, 2:10].stack().mean() + nxtab_flhsc['Node']['Gene'].iloc[:, 2:10].stack().std()
rmin = nxtab_flhsc['Node']['Region'].iloc[:, 2:10].stack().mean() - nxtab_flhsc['Node']['Region'].iloc[:, 2:10].stack().std()
rmax = nxtab_flhsc['Node']['Region'].iloc[:, 2:10].stack().mean() + nxtab_flhsc['Node']['Region'].iloc[:, 2:10].stack().std()

In [ ]:
G, pos, edge_tables, node_tables = create_nx_graph(nxtab_flhsc,
                   use_edge_tables = ['TF2R','R2G'],
                   color_edge_by = {'TF2R': {'variable' : 'TF', 'category_color' : category_color_flhsc},
                                    'R2G': {'variable' : 'rho_R2G', 'continuous_color' : 'viridis', 'v_min': -1, 'v_max': 1}},
                   #transparency_edge_by =  {'R2G': {'variable' : 'importance_R2G', 'min_alpha': 0.1, 'v_min': 0}},
                   width_edge_by = {'R2G': {'variable' : 'importance_R2G', 'max_size' :  1.5, 'min_size' : 1}},
                   color_node_by = {'TF': {'variable': 'TF', 'category_color' : category_color_flhsc},
                                    'Gene': {'variable': 'scenic_ident_Log2FC_FLHSC', 'continuous_color' : 'bwr', 'v_min': gmin, 'v_max': gmax},
                                    'Region': {'variable': 'scenic_ident_Log2FC_FLHSC', 'continuous_color' : 'viridis', 'v_min': rmin, 'v_max': rmax}},
                   #transparency_node_by =  {'Region': {'variable' : 'scenic_ident_Log2FC_FLHSC', 'min_alpha': 0.1},
                   #                 'Gene': {'variable' : 'scenic_ident_Log2FC_FLHSC', 'min_alpha': 0.1}},
                   size_node_by = {'TF': {'variable': 'fixed_size', 'fixed_size': 20},
                                    'Gene': {'variable': 'fixed_size', 'fixed_size': 10},
                                    'Region': {'variable': 'fixed_size', 'fixed_size': 8}},
                   shape_node_by = {'TF': {'variable': 'fixed_shape', 'fixed_shape': 'ellipse'},
                                    'Gene': {'variable': 'fixed_shape', 'fixed_shape': 'ellipse'},
                                    'Region': {'variable': 'fixed_shape', 'fixed_shape': 'diamond'}},
                   label_size_by = {'TF': {'variable': 'fixed_label_size', 'fixed_label_size': 9.0},
                                    'Gene': {'variable': 'fixed_label_size', 'fixed_label_size': 5.0},
                                    'Region': {'variable': 'fixed_label_size', 'fixed_label_size': 1.0}},
                   layout='kamada_kawai_layout',
                   scale_position_by=250)

scenicplus.networks.export_to_cytoscape(G, pos, out_file = outDir + 'cs_fl_hsc.cyjs')

In [ ]:
scenicplus.networks.plot_networkx(G, pos)

In [ ]:
print(nxtab_flhsc['Node']['Gene'].iloc[:, 7])
gvals = nxtab_flhsc['Node']['Gene'].iloc[:, 7]
data = (gvals, gvals)

cmap = plt.cm.get_cmap("bwr")
plt.imshow(data, cmap, vmin = gmin, vmax = gmax)
plt.colorbar()
plt.savefig(outDir + "fl_hsc_genecolors.pdf", format = "pdf")  

In [ ]:
print(nxtab_flhsc['Node']['Region'].iloc[:, 7])
gvals = nxtab_flhsc['Node']['Region'].iloc[:, 7]
data = (gvals, gvals)

cmap = plt.cm.get_cmap("viridis")
plt.imshow(data, cmap, vmin = rmin, vmax = rmax)
plt.colorbar()
plt.savefig(outDir + "fl_hsc_regioncolors.pdf", format = "pdf")  